#### To build a human ME model, 3 inputs are needed:

1) a cobrapy model in .json format. This most likely will need to be recon2.2 or a context-extracted version of recon2.2, as this is what the code was built on that model and has not been tested on other models

2) a list of non-machinery proteins (ie, proteins not involved in catalyzing any of the metabolic module or expression module reactions) in .txt format; this can be empty if you do not wish to express any non-machinery proteins -- this part is not completed yet

3) a protein specific information matrix (PSIM) in .csv format. For each gene in the input model, the PSIM should contain the following columns: HGNC_ID, PREMRNA_SEQ, MRNA_SEQ, PROTEIN_SEQ, POLYA_LENGTH, TMD, SP, N_INTRONS, DSB, GPI, OG, LOCATION. The first four columns are required, the remaining are optional (can be nan or None values instead). LOCATION is required for non-machinery proteins, and must be in list format. The allowed compartments are: 'c', 'l', 'm', 'r', 'e', 'x', 'n', 'g', 'i', 'pm'. The first four columns must be strings; details on limitations can be found by looking at the gene_information class. PREMRNA_SEQ is assumed to contain the 5' UTR, 3' UTR and introns, whereas MRNA_SEQ does not contain introns. POLYA_LENGTH must be a float which represents the length of the polyA tail. TMD must be an integer >= 0 representing the number of transmembrane domains in the protein. SP is a Boolean representing the presence of a signal peptide; in the current version of the code, this is overwritten by the location. N_INTRONS is an integer representing the number of introns in the transcript. DSB, GPI, and OG are all post-translational modifications corresponding to disulfide bond formation, GPI anchor, and O-glycosylation (in the current version, only considered for non-machinery; N-glycosylation will be added later). The values for these must be integers corresponding to the number of each type of PTM the protein receives. For GPI, this value should be 0 for presence and 1 for absence.




---------
Before starting this guide, make sure to run the correct_inputs.py script, which will check and change input formatting, and store them in the processed directory within local_data_path. utils.py loads these re-written files, so this is a necessary step. Similarly run the download_data.py script

In [1]:
import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from utils.load_environmental_variables import *

Ribosome biogenesis and trna reactions are hardcoded

In [3]:
# ribosomal_reactions is a list of cobra.Reactions corresponding to ribosome biogenesis
# ribosome_complex_c is a cobra.Metabolite corresponding to the cytoplasmic ribosome
from uniform_processes.build_ribosome_biogenesis_reactions import ribosomal_reactions, ribosome_complex_c

# similarly, trna_biogenesis reactions is a list of cobra.Reactions corresponding to trna biogenesis
# charged_trna_metabolites is a list of cobra.Metabolites corresponding to a charged trna for each amino acid
# modified_trna_transcript_c is the universal uncharged trna cobra.Metabolite
from uniform_processes.build_trna_expression_reactions import trna_biogenesis_reactions, charged_trna_metabolites, modified_trna_transcript_c

### To build your own protein expression reaction

Use the following code. Let's take a toy example gene with information as represented in the PSIM

In [4]:
import random
import pandas as pd
from utils import parameters as params
psim_toy = pd.DataFrame(columns = ['HGNC_ID', 'PREMRNA_SEQ', 'MRNA_SEQ', 'PROTEIN_SEQ', 'POLYA_LENGTH', 'TMD', 
                               'SP', 'N_INTRONS', 'DSB', 'GPI', 'OG', 'LOCATION'])

hgnc_id, premrna_seq = 'HGNC:TOY', ''.join(random.choices(['U', 'C', 'G', 'A'], k = 100))
mrna_seq = premrna_seq[25:75]
# note that there is no check that the protein_sequence corresponds to the mrna_sequence beyond checking for the length
protein_seq = ''.join(random.choices(params.amino_acids, k = int(len(mrna_seq)/3)))
polyA_length, tmd, sp, n_introns, dsb, gpi, og  = None, 1, True, 0, 2, 2, 2
location = ['c'] # cytoplasm and golgi

psim_toy.loc[0,:] = [hgnc_id, premrna_seq, mrna_seq, protein_seq, polyA_length, tmd, sp, n_introns, dsb, gpi, og, location]

In [5]:
psim_toy.head()

,HGNC_ID,PREMRNA_SEQ,MRNA_SEQ,PROTEIN_SEQ,POLYA_LENGTH,TMD,SP,N_INTRONS,DSB,GPI,OG,LOCATION
0,HGNC:TOY,AUUGGUGAUCAGGUUCCCGAUUGAGGCUACAGAGUGCACACGGCUG...,GCUACAGAGUGCACACGGCUGAGAGGAAUCAGCUGGAAUAUUGAUC...,IFMWFFKGFQYCTWIC,None,1,True,0,2,2,2,[c]


The gene information class stores all the extrapolates and stores all relevant information necessary for building the expression reactions from the PSIM. 

In [7]:
from expression.gene_information import gene_information

There are two ways to generate a gene_information object. 

#### Method 1: Manual generation

In [8]:
# metabolic_machinery is a list of all the genes in the input cobrapy model
# utils makes this default to list(model.genes), see utils.metabolic_machinery
# this checks whether the protein is metabolic machinery or non-machinery, and assigns it to .module
# there is also an internal check to see whether the gene is expression machinery
gene_info = gene_information(hgnc_id, premrna_seq, mrna_seq, protein_seq,
                 ptms = {'dsb': dsb, 'og': og, 'gpi': gpi}, tmd = tmd, sp = sp, polyA_length = polyA_length, 
                 n_introns = n_introns)
gene_info.module

'Non-Machinery'

Next, we add the final locations. If this was a machinery protein, the user input would be overwritten by the compartment extracted from the model


In [9]:
import cobra
gene_info.get_final_locations(metabolic_model = cobra.Model(''), final_locations = location)
gene_info.final_locations

../scripts/expression/gene_information.py:260 UserWarning: HGNC:TOY: Signal peptides not considered for cytosol


{'c': 'Cytosolic Tranport'}

Finally, we check that all the inputs make sense to build the expression reactions, and correct any mistaken inputs

In [10]:
gene_info.check_gene_information()

No errors raised


../scripts/expression/gene_information.py:298 UserWarning: HGNC:TOY: GPI is binary, 1 for presence or 0 for absence. Changing to 1


Alternatively, we can create the gene_information object directly from the PSIM

#### Method 2: Generate from PSIM

In [11]:
from utils import utils_2
gene_info = utils_2.generate_geneinfo_object(hgnc_id, psim = psim_toy, metabolic_model = cobra.Model(''))

No errors raised


Now, this is all the input needed to create the expression reactions.

In [12]:
import expression.build_mrna_expression_reactions as build_mrna
import expression.build_protein_expression_reactions as build_protein

# mrna_reactions is a list of cobra.Reactions including transcript elongation, processing, transport, and degradation
# mrna_transcript_c is the mature, cytoplasmic mrna transcript cobra.Metabolite
mrna_reactions, mrna_transcript_c, mrna_deg_proxy = build_mrna.get_mrna_expression_reactions(gene_info)

In [13]:
mrna_reactions

[<Reaction HGNC:TOY_TRANSCRIPTION_ELONGATION at 0x7f8a0edb2cc0>,
 <Reaction HGNC:TOY_lariats_DEGRADATIONn at 0x7f8a117a32e8>,
 <Reaction HGNC:TOY_TRANSCRIPTION_PROCESSING at 0x7f8a117a3278>,
 <Reaction HGNC:TOY_mRNA_EXPORTtn at 0x7f8a0edb29e8>,
 <Reaction HGNC:TOY_DECAPPING_mRNA_DEGRADATIONc at 0x7f8a11758a20>]

In [15]:
# ub reactions is a list of ubiquitin degradation reactions independent of gene_info
# which are necessary for degradation of all cytoplasmic, nuclear, ER, and golgi proteins
ub_reactions = build_protein.ub_reactions

# similarly, protein_reactions is a list of cobra.Reactions including translation elongation, folding, degradation and transport to the appropriate final locations
# protein metabolites is a list of the mature proteins in each final location
protein_reactions, protein_metabolites = build_protein.get_protein_expression_reactions(gene_info, mrna_transcript_c, mrna_deg_proxy)

In [16]:
protein_reactions

[<ME_Reaction HGNC:TOY_TRANSLATION_ELONGATIONc at 0x7f8a0edb25c0>,
 <Reaction HGNC:TOY_CYTOSOLIC_PROTEIN_FOLDING at 0x7f8a117657f0>,
 <Reaction HGNC:TOY_folded_protein[c]_POLYUBIQUITINATIONc at 0x7f8a11765828>,
 <Reaction HGNC:TOY_folded_protein[c]_DEUBIQUITINATIONc at 0x7f8a11765a90>,
 <Reaction HGNC:TOY_folded_protein[c]_PROTEASOMAL_DEGRADATIONc at 0x7f8a117657b8>]

Ofcourse, in a ME model, the machinery should be added as a substrate in the reaction itself, including the expression reactions. The build_me_model.py script does this for all machinery and reactions in the input model and input non-machinery. 

### To build a ME Model

Will change this behavior in the future to take in inputs, but for now, the input files are specified in the .env file. 

After this is done, simply run master.py script and the output ME model will be saved as a cobrapy json file. Also will change this to be stored as a specified output file. For now, it will be saved in the 'processed' subdirectory within the specified local data path, under the name 'human_me_model.json'
